<a href="https://colab.research.google.com/github/T-Svitlichna/DTA_Python/blob/main/HomeWork/ML_23062026_ml_practice_LR%26Cl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика: лінійна регресія та класифікація

Це тренувальний блокнот для закріплення базового циклу ML. Завдання **нескладні** й повторюють кроки з основного тьюторіалу — тільки тепер усе робиш **сам**.

**Дві задачі на двох нових наборах даних:**
- **Задача A (регресія):** передбачити **зарплату** працівника.
- **Задача B (класифікація):** передбачити, чи **складе студент іспит** (так/ні).

**Як працювати:**
1. Запусти комірку «Підготовка даних» нижче — вона все налаштує.
2. Іди по кроках. Там, де стоїть `# TODO`, — впиши свій код.
3. Підказки є під кожним кроком.

> 💡 Усі потрібні інструменти ти вже бачив: `train_test_split`, `LinearRegression`, `DecisionTreeClassifier`, `.fit()`, `.predict()`, метрики. Тримай той блокнот поруч як шпаргалку.

---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)

# ---------- Дані A: ЗАРПЛАТИ (для регресії) ----------
N = 800
experience = np.random.randint(0, 31, N)                      # стаж, років
age        = (22 + experience + np.random.randint(0, 12, N)).clip(22, 64)  # вік
education  = np.random.choice([12, 15, 16, 18, 20], N,        # років освіти
                              p=[.2, .15, .35, .2, .1])       # # ← ЙМОВІРНОСТІ (як часто)
english    = np.random.randint(1, 6, N)                       # рівень англ. 1..5

salary = (8000                       # базова ставка, грн
          + experience * 900         # за кожен рік стажу
          + education  * 600         # за рік освіти
          + english    * 1500        # за рівень англійської
          + np.random.normal(0, 3000, N)   # шум: усе інше
         ).clip(8000, None)

salary_df = pd.DataFrame({
    "experience": experience, "age": age,
    "education": education, "english": english,
    "salary": salary.round(0).astype(int),
})

# ---------- Дані B: СТУДЕНТИ (для класифікації) ----------
M = 800
study     = np.random.normal(12, 5, M).clip(0, 30)            # годин навчання/тиждень
attendance= np.random.normal(78, 15, M).clip(30, 100)        # відвідуваність, %
prev_score= np.random.normal(65, 18, M).clip(0, 100)         # бал за минулий іспит
sleep     = np.random.normal(7, 1.2, M).clip(4, 10)          # годин сну

score_logit = (0.12*study + 0.04*attendance + 0.05*prev_score
               + 0.3*sleep - 9 + np.random.normal(0, 1.2, M))
passed = (score_logit > 0).astype(int)                        # 1 = склав, 0 = ні

students_df = pd.DataFrame({
    "study": study.round(1), "attendance": attendance.round(0).astype(int),
    "prev_score": prev_score.round(0).astype(int), "sleep": sleep.round(1),
    "passed": passed,
})

print("✅ Дані готові.")
print("Зарплати:", salary_df.shape, "| Студенти:", students_df.shape)
print("Частка тих, хто склав іспит:", f"{students_df['passed'].mean():.0%}")

✅ Дані готові.
Зарплати: (800, 5) | Студенти: (800, 5)
Частка тих, хто склав іспит: 69%


---
# 🟦 Задача A. Регресія: передбачаємо зарплату

Дані у таблиці `salary_df`. Ознаки: `experience` (стаж), `age` (вік), `education` (років освіти), `english` (рівень англійської 1–5). Ціль: `salary` (зарплата, грн).

Мета — навчити модель передбачати зарплату і **пояснити**, що на неї впливає.

### Крок A1. Подивись на дані
Виведи перші рядки таблиці й описову статистику. Це звичка №1 перед будь-яким навчанням.

*Підказка:* `salary_df.head()` і `salary_df.describe()`.

In [2]:
# TODO: виведи перші рядки salary_df
salary_df.head()
# TODO: виведи describe()
salary_df.describe()

,experience,age,education,english,salary
count,800.000000,800.000000,800.000000,800.000000,800.000000
mean,15.418750,42.746250,15.856250,3.028750,36028.926250
std,9.328568,9.924338,2.408908,1.432826,9242.043432
min,0.000000,22.000000,12.000000,1.000000,12917.000000
25%,7.000000,35.000000,15.000000,2.000000,28550.750000
50%,16.000000,43.000000,16.000000,3.000000,36102.500000
75%,24.000000,51.000000,18.000000,4.000000,43545.500000
max,30.000000,62.000000,20.000000,5.000000,55947.000000


### Крок A2. Признач ознаки (X) і ціль (y), поділи на train / test
- `X` — усі стовпці, КРІМ `salary`.
- `y` — стовпець `salary`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`.

*Підказка:* `X = salary_df[["experience", "age", "education", "english"]]`,
`y = salary_df["salary"]`, далі `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)`.

In [3]:
from sklearn.model_selection import train_test_split

# TODO: створи X та y
X = salary_df[["experience", "age", "education", "english"]]
y = salary_df["salary"]

# TODO: поділи на X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=RANDOM_STATE)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 640 | Test: 160


### Крок A3. Навчи лінійну регресію
Згадай цикл: **створити → `.fit(X_train, y_train)`**.

*Підказка:* `from sklearn.linear_model import LinearRegression`, далі `model = LinearRegression()` і `model.fit(...)`.

In [4]:
from sklearn.linear_model import LinearRegression
model1 = LinearRegression()
model1.fit(X_train, y_train)
y_pred = model1.predict(X_test) #передбачення на тестовими значеннями

# TODO: створи та навчи модель

### Крок A4. Зроби передбачення й оціни якість
- Передбач на `X_test`.
- Порахуй **MAE** та **R²**.

*Підказка:* `y_pred = model.predict(X_test)`; `mean_absolute_error(y_test, y_pred)`;
`r2_score(y_test, y_pred)`.

In [5]:
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"R²: {r2*100:.3f} model describe % from real salary")
print(f"MAE: {mae:.1f}K  in mean error")

# TODO: порахуй і виведи MAE та R²

R²: 88.145 model describe % from real salary
MAE: 2545.3K  in mean error


### Крок A5. 🔑 Інтерпретуй коефіцієнти
Дістань коефіцієнти моделі й скажи словами, яка ознака найсильніше підвищує зарплату.

*Підказка:* `model.coef_` і `model.intercept_`. Зістав назви з `X.columns`.

In [6]:
# TODO: побудуй таблицю "ознака — коефіцієнт" і відсортуй
coef_salary = pd.DataFrame({
    "ознака": X.columns,
    "коефіцієнт": model1.coef_.round(0)
}).sort_values("коефіцієнт", ascending=False)

print(coef_salary)
print(f"\nIntercept: {model1.intercept_:.2f}")

       ознака  коефіцієнт
3     english      1597.0
0  experience       872.0
2   education       610.0
1         age        31.0

Intercept: 6850.03


✍️ **Запиши відповідь словами** (просто текстом у цій комірці, подвійний клік):
> Найсильніше на зарплату впливає ознака - English, наявність знань з англійської додає до ЗП + майже 1600 грн.

### Крок A6. Передбач зарплату для нового працівника
Створи одного працівника й передбач його зарплату: стаж 5, вік 30, освіта 16, англійська 4.

*Підказка:* зроби `pd.DataFrame([{...}])` з тими самими назвами стовпців і передай у `model.predict(...)`.

In [7]:
# TODO: створи new_employee і передбач зарплату
new_employee1 = pd.DataFrame({
    "experience": [5],
    "age": [30],
    "education": [16],
    "english": [4]
})
model1.predict(new_employee1)
print(f"Зарплата нового працівника 1: {model1.predict(new_employee1)[0]:.0f} грн")

Зарплата нового працівника 1: 28290 грн


In [8]:
new_employee2 = pd.DataFrame({
    "experience": [0],
    "age": [18],
    "education": [11],
    "english": [10]
})
model1.predict(new_employee2)
print(f"Зарплата нового працівника 2: {model1.predict(new_employee2)[0]:.0f} грн")

Зарплата нового працівника 2: 30091 грн


In [9]:
new_employee3 = pd.DataFrame({
    "experience": [0],
    "age": [65],
    "education": [16],
    "english": [0]
})
model1.predict(new_employee3)
print(f"Зарплата нового працівника 3: {model1.predict(new_employee3)[0]:.0f} грн")

Зарплата нового працівника 3: 18628 грн


---
# 🟩 Задача B. Класифікація: чи складе студент іспит

Дані у таблиці `students_df`. Ознаки: `study` (годин навчання/тиждень), `attendance` (відвідуваність %), `prev_score` (бал за минулий іспит), `sleep` (годин сну). Ціль: `passed` (1 = склав, 0 = ні).

### Крок B1. Подивись на дані
Виведи перші рядки й перевір баланс класів: яка частка студентів склала іспит?

*Підказка:* `students_df.head()` і `students_df["passed"].mean()`.

In [10]:
# TODO: head() і частка тих, хто склав
students_df.head()
print(f"Частка тих, хто склав іспит: {students_df['passed'].mean():.0%}")

Частка тих, хто склав іспит: 69%


### Крок B2. X, y і поділ на train / test
- `X` — усе, крім `passed`. `y` — `passed`.
- Додай `stratify=y`, щоб пропорція класів збереглася.

*Підказка:* `train_test_split(Xs, ys, test_size=0.2, random_state=RANDOM_STATE, stratify=ys)`.

In [11]:
# TODO: Xs, ys та поділ на Xs_train, Xs_test, ys_train, ys_test
Xs = students_df[["study", "attendance", "prev_score", "sleep"]]
ys = students_df["passed"]
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=RANDOM_STATE, stratify=ys)

print("Train:", Xs_train.shape[0], "| Test:", Xs_test.shape[0])

Train: 640 | Test: 160


### Крок B3. Навчи дерево рішень
Використай `DecisionTreeClassifier` з `max_depth=3` (щоб було просте й читабельне) і `random_state=RANDOM_STATE`.

*Підказка:* `from sklearn.tree import DecisionTreeClassifier`.

In [12]:
from sklearn.tree import DecisionTreeClassifier
model2 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
model2.fit(Xs_train, ys_train)
y_pred = model2.predict(Xs_test)



# TODO: створи та навчи дерево

### Крок B4. Передбач і оціни
- Передбач на `Xs_test`.
- Порахуй **accuracy** і побудуй **матрицю плутанини**.

*Підказка:* `accuracy_score(ys_test, ys_pred)` та `confusion_matrix(ys_test, ys_pred)`.

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix
accs = accuracy_score(ys_test, y_pred)
print(f"Accuracy: {accs*100:.2f}%")
con_matrix = confusion_matrix(ys_test, y_pred)
con_matrix = pd.DataFrame(con_matrix,
    index=["Факт: не склав", "Факт: склав"],
    columns=["Передбачено: не склав", "Передбачено: склав"])
print(confusion_matrix(ys_test, y_pred))
print(con_matrix)
# TODO: передбач ys_pred, порахуй accuracy та матрицю плутанини

Accuracy: 75.62%
[[25 24]
 [15 96]]
                Передбачено: не склав  Передбачено: склав
Факт: не склав                     25                  24
Факт: склав                        15                  96


додаткові перевірки моделі, матриця показала погані результати для передбачення "не скдав".

In [14]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(ys_test, y_pred,
      target_names=["не склав", "склав"]))

y_proba = model2.predict_proba(Xs_test)[:, 1]
print(f"ROC AUC: {roc_auc_score(ys_test, y_proba):.3f}")

              precision    recall  f1-score   support

    не склав       0.62      0.51      0.56        49
       склав       0.80      0.86      0.83       111

    accuracy                           0.76       160
   macro avg       0.71      0.69      0.70       160
weighted avg       0.75      0.76      0.75       160

ROC AUC: 0.762


In [15]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model2, Xs_train, ys_train, cv=5)
print(f"CV scores: {scores.round(3)}")
print(f"Середнє:   {scores.mean():.3f} ± {scores.std():.3f}")

CV scores: [0.711 0.789 0.719 0.758 0.742]
Середнє:   0.744 ± 0.028


### Крок B5. Що найбільше впливає на результат?
Виведи важливість ознак дерева й назви найважливішу.

*Підказка:* `tree.feature_importances_`, зістав із `Xs.columns`.

In [16]:
# TODO: таблиця "ознака — важливість", відсортована за спаданням
importances = pd.DataFrame({
    "ознака": Xs.columns,
    "важливість": model2.feature_importances_
}).sort_values("важливість", ascending=False)
print(importances)

       ознака  важливість
2  prev_score    0.377026
0       study    0.325006
1  attendance    0.195933
3       sleep    0.102035


✍️ **Відповідь словами:**
> Найбільше на складання іспиту впливає бал за минулий іспит.

### Крок B6. Передбач для нового студента
Студент: навчання 15 год, відвідуваність 85%, минулий бал 70, сон 7.5.
Виведи і рішення (`predict`), і **ймовірність** скласти (`predict_proba`).

*Підказка:* `predict_proba(...)[0, 1]` — це ймовірність класу «склав».

In [17]:
# TODO: створи new_student, виведи рішення та ймовірність
new_student = pd.DataFrame({
    "study": [15],
    "attendance": [85],
    "prev_score": [70],
    "sleep": [7.5]
})
print(f"Розв'язок: {model2.predict(new_student)[0]}")
print(f"Ймовірність: {model2.predict_proba(new_student)[0, 1]:.3f}")

Розв'язок: 1
Ймовірність: 0.881


---
# ⭐ Бонус (необов'язково, але корисно)

1. **Перевір на перенавчання.** Для дерева зі Задачі B порахуй accuracy окремо на `Xs_train` і на `Xs_test`. Великий розрив = зубріння. Потім спробуй `max_depth=10` — розрив зросте?
2. **Сильніша модель.** Навчи `RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)` на тих самих даних і порівняй accuracy з деревом.
3. **Прибери ознаку.** У Задачі A прибери `experience` з `X`, перенавчи й подивись, як впаде R². Який висновок про важливість стажу?

In [20]:
# Місце для бонусних експериментів
X_no_exp = salary_df[["age", "education", "english"]]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_no_exp, y, test_size=0.2, random_state=RANDOM_STATE)

model4 = LinearRegression()
model4.fit(X_tr, y_tr)
r2_mod4 = r2_score(y_te, model4.predict(X_te))

print(f"R² з experience:    {r2_score(y_test, model1.predict(X_test)):.3f}")
print(f"R² без experience:  {r2_mod4:.3f}")
print(f"Падіння R²: -{r2_score(y_test, model1.predict(X_test)) - r2_mod4:.3f}")
# Досвід є важливим показником

R² з experience:    0.881
R² без experience:  0.767
Падіння R²: -0.115


---
# 🧠 Питання на розуміння (без коду)

Дай собі відповідь словами:
1. Чому ми оцінюємо модель на `X_test`, а не на `X_train`?
2. Задача «передбачити кількість проданих квитків» — це регресія чи класифікація? А «спам / не спам»?
3. Що означає R² = 0.85 простими словами?
4. Чому accuracy може бути оманливою, якщо лише 5% студентів провалюють іспит?
5. Коефіцієнт `english = +1500`. Як прочитати це вголос для керівника?


✍️ **Відповідь словами:**

1.  Ми оцінюємо модель на тестових значення, для того щоб модель не скопіювала повністю логіку X_train та не видала нам підготовлені заздалегіть відповіді.
2. Кількісні значення - це регресія, а відповіді так чи ні - це класифікація.
3. R² - показує наскільки наша модель навчана, скількі відсотків даннних вона вірно повертає(вгадує/знайшла вірні взаємозвязки) - 0,85 із 1 - це 85%. Добрий показник
4. Ці 5 відсотків модель може вважати як похибка та не брати йх до уваги і виключити його із звіту, проігнорувати.
5. Кожний новий рівень англійської мови підвищує зарплату працівника на 1500 у.о.

> 🎯 Якщо впорався із задачами A і B без ШІ — ти впевнено володієш базовим циклом ML. Вітаю!